# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # Replace with your OpenAI API key (or set from another env var)
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint

import pandas as pd

import importlib
import hypothesaes.quickstart
importlib.reload(hypothesaes.quickstart)
from hypothesaes.quickstart import interpret_sae

import hypothesaes.embedding 
importlib.reload(hypothesaes.embedding)
from hypothesaes.embedding import get_local_embeddings

from hypothesaes.quickstart import train_sae

INTERPRETER_MODEL = "openai/gpt-oss-20b"
ANNOTATOR_MODEL = INTERPRETER_MODEL
# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
ANNOTATION_LLM_KWARGS = {"reasoning_effort": "low"}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits

**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
is_vast = True
if is_vast:
    train_path = os.path.join(prefix, 'data', 'vast_train.csv')
    val_path = os.path.join(prefix, 'data', 'vast_dev.csv')
    DOC_FIELD = 'post'
    TOPIC_FIELD = 'new_topic'
else:
    train_path = os.path.join(prefix, 'data', 'raw_train_all_onecol.csv')
    train_path = os.path.join(prefix, 'data', 'raw_val_all_onecol.csv')
    DOC_FIELD = "Ori Text"
    TOPIC_FIELD = "Target 1"
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
all_train_texts = list(train_df[DOC_FIELD])
all_train_topics = list(train_df[TOPIC_FIELD])
unique_train_texts  = list(train_df[DOC_FIELD].unique())
unique_train_topics = list(train_df[TOPIC_FIELD].unique())


In [3]:
all_val_texts  = list(val_df[DOC_FIELD])
all_val_topics = list(val_df[TOPIC_FIELD])
val_df = val_df[~(
    val_df[DOC_FIELD].map(set(unique_train_texts).__contains__) |
    val_df[TOPIC_FIELD].map(set(unique_train_topics).__contains__)
)]
unique_val_texts  = list(val_df[DOC_FIELD].unique())
unique_val_topics = list(val_df[TOPIC_FIELD].unique())

In [4]:
embed_func = lambda texts: get_local_embeddings(texts, return_stacked=True)
unique_train_text_embeds  = embed_func(unique_train_texts)
unique_val_text_embeds    = embed_func(unique_val_texts)
unique_train_topic_embeds = embed_func(unique_train_topics)
unique_val_topic_embeds   = embed_func(unique_val_topics)

all_train_text_embeds = embed_func(all_train_texts)
all_val_text_embeds = embed_func(all_val_texts)
all_train_topic_embeds = embed_func(all_train_topics)
all_val_topic_embeds = embed_func(all_val_topics)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


... (22 outputs left out)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/17 [00:00<?, ?it/s]

In [5]:
shared_kwargs = {
    "M": 256,
    "K": 8,
    "matryoshka_prefix_lengths": [32, 256],
    "n_epochs": 300
}
text_sae = train_sae(embeddings=unique_train_text_embeds, val_embeddings=unique_val_text_embeds, **shared_kwargs)
topic_sae = train_sae(embeddings=unique_train_topic_embeds, val_embeddings=unique_val_topic_embeds, **shared_kwargs)
# text_sae = train_sae(embeddings=all_train_text_embeds, val_embeddings=all_val_text_embeds, **shared_kwargs)
# topic_sae = train_sae(embeddings=all_train_topic_embeds, val_embeddings=all_val_topic_embeds, **shared_kwargs)

  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 117 epochs


  0%|          | 0/300 [00:00<?, ?it/s]

Early stopping triggered after 111 epochs


In [6]:
import numpy as np
def get_labels(df):
    return np.array(list(map(int, df['label'])))
train_labels = get_labels(train_df)
val_labels = get_labels(val_df)

In [29]:
lukasiewicz = lambda A, B: np.maximum(0, A + B - 1)

In [34]:
import numpy as np
def mono_acts(texts, topics):
    text_embeds = embed_func(texts)
    topic_embeds = embed_func(topics)
    text_acts = text_sae.get_activations(text_embeds)
    topic_acts = topic_sae.get_activations(topic_embeds)
    return text_acts, topic_acts
def conj_acts(A, B, conj=np.minimum):
    n_samples = A.shape[0]
    conj_acts = conj(np.expand_dims(A, 2), np.expand_dims(B, 1))
    flattened = np.reshape(conj_acts, (n_samples, -1))
    return flattened
def features_from_df(df, with_mono=False):
    mono_text, mono_topic = mono_acts(df[DOC_FIELD], df[TOPIC_FIELD])
    features = conj_acts(mono_text, mono_topic)
    if with_mono:
        features = np.concatenate([mono_text, mono_topic, features], axis=1)
    return features
with_mono = True
val_acts = np.concat(mono_acts(val_df[DOC_FIELD], val_df[TOPIC_FIELD]), axis=1)
train_acts = np.concat(mono_acts(train_df[DOC_FIELD], train_df[TOPIC_FIELD]), axis=1)
# val_acts   = features_from_df(val_df, with_mono=with_mono)
# train_acts = features_from_df(train_df, with_mono=with_mono)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Loaded model nomic-ai/modernbert-embed-base to cuda


... (10 outputs left out)

Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Chunk 0:   0%|          | 0/106 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
from collections import Counter
c = Counter(train_labels.tolist())
class_counts = np.array([c[i] for i in range(3)])
class_weights = len(c) / (3 * class_counts)
train_and_val_labels = np.concat([train_labels, val_labels])
train_and_val_weights = class_weights[train_and_val_labels]

In [35]:
from cuml.linear_model import LogisticRegression
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import precision_recall_fscore_support
split = PredefinedSplit(test_fold=np.concat([np.full_like(train_labels, -1), np.zeros_like(val_labels)]))
param_grid = {
    "l1_ratio": [0.99],
}
cv = GridSearchCV(
    estimator=LogisticRegression(solver='qn'),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=split,
)
fitted_clf = cv.fit(
    np.concat([train_acts, val_acts]),
    train_and_val_labels,
    sample_weight=train_and_val_weights
)
val_preds = fitted_clf.predict(val_acts)
res = precision_recall_fscore_support(val_labels, val_preds)
res

(array([0.46292948, 0.45840708, 0.31818182]),
 array([0.65641026, 0.67801047, 0.03589744]),
 array([0.54294804, 0.5469905 , 0.06451613]),
 array([390, 382, 390]))

In [36]:
metric_by_class = np.stack(res[:3], axis=0)
macro_avgs = np.mean(metric_by_class, axis=1)
macro_avgs

array([0.41317279, 0.45677272, 0.38481822])

In [10]:
topic_task_instructions = """All of the below samples are topics discussed on online fora.
Features should be more general than the topics themselves."""
# Interpret random neurons
results_topics = interpret_sae(
    texts=unique_train_topics,
    embeddings=unique_train_topic_embeds,
    # texts=all_train_topics,
    # embeddings=all_train_topic_embeds,
    sae=topic_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=topic_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (5015, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 156 (1.5% active): mentions the word 'politics'

Top activating examples:
1. politics
2. political
3. political thought
----------------------------------------------------------------------------------------------------

Neuron 49 (1.6% active): mentions the word 'mexican'

Top activating examples:
1. mexican
2. mexico
3. latino
----------------------------------------------------------------------------------------------------

Neuron 163 (2.4% active): mentions alliance

Top activating examples:
1. working together
2. allying with enemies in war
3. countries working together on climate change issues
----------------------------------------------------------------------------------------------------

Neuron 152 (2.3% active): mentions disaster

Top activating examples:
1. disaster
2. housing market crash
3. flag burning
----------------------------------------------------------------------------------------------------

Neuron 144 (1.9% active): mentions law

Top activating e

In [11]:
text_task_instructions = """All of the below samples are posts from an online forum.
Features should describe a specific aspect of the post (especially what topics it discusses)."""
# Interpret random neurons
results = interpret_sae(
    texts=unique_train_texts,
    embeddings=unique_train_text_embeds,
    sae=text_sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=text_task_instructions,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (1845, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 212 (3.4% active): mentions Coursera

Top activating examples:
1. Innovation, a concrete concept, was once linked to Progress, an abstract concept. This is no longer the case. We live in a world in which the main goal of innovators is to disrupt our lives by forcing us to by new stuff. As Edward Abbey said, "Growth for the sake of growth is the ideology of a cancer cell."
2. The unacknowledged advantage of privileging 'starchitects'' over the local talents is that the first come with a set of standards, a rigorous vocabulary and a name that prevail over the compromises usually inherent in a developer-driven culture (cost cuts, value-engineering, the client's 'taste', etc. - whatever could damage a project from the design phase to the actual built outcome). If one looks objectively, disregarding personal taste, whether it is a Gehry, a Nouvel or Isozaki building here in New York - they all stand out as impeccably executed architecture. Most of them relate to their context (see t